1. Install update and run software tests


In [ ]:
from google.colab import files, drive
from pathlib import Path
import subprocess, sys, tempfile, zipfile

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/scientific-news-verification')
uploaded = files.upload()
matches = []
for name in uploaded:
    if zipfile.is_zipfile(name):
        with zipfile.ZipFile(name) as archive:
            if 'install_connected_update.py' in archive.namelist():
                matches.append(name)
assert len(matches) == 1, 'Upload Scientific_News_Connected_Development_Update.zip.'
bundle = Path(tempfile.mkdtemp(prefix='connected_update_'))
with zipfile.ZipFile(matches[0]) as archive:
    for item in archive.infolist():
        target = (bundle / item.filename).resolve()
        assert target.is_relative_to(bundle.resolve()), 'Unsafe archive path.'
    archive.extractall(bundle)
subprocess.run([sys.executable, str(bundle/'install_connected_update.py'), '--project', str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=PROJECT, check=True)


2. Run one live E2 assertion check


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys

PROJECT = Path('/content/drive/MyDrive/scientific-news-verification')
BASE = PROJECT/'private/eligibility_screening/batch_20261119T080435821348Z/public_fulltext_screen_20261119T081014931038Z'
CASE = BASE/'train14460_condition_inputs_20261119T091631988786Z'
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = PROJECT/f'private/connected_claim_runs/train14460_{stamp}'
print('One live model generation; no automatic retries.', flush=True)
print('Results:', OUT, flush=True)
subprocess.run([sys.executable, 'connected_claim_runner_v1.py', '--project', str(PROJECT), '--case', str(CASE), '--xml', str(BASE/'xml/train_14460.xml'), '--output', str(OUT), '--live'], cwd=PROJECT, check=True)


3. Screen remaining 270 pairs; resume this same cell after interruption


In [ ]:
from pathlib import Path
import subprocess, sys

PROJECT = Path('/content/drive/MyDrive/scientific-news-verification')
BATCH = PROJECT/'private/eligibility_screening/batch_20261119T080435821348Z'
OUT = BATCH/'remaining_public_screen_v1'
print('Screening remaining 270 records. Saved records are skipped on resume.', flush=True)
subprocess.run([sys.executable, 'screen_remaining_v1.py', '--metadata', str(BATCH/'screening_metadata.csv'), '--manifest', str(BATCH/'screening_manifest.json'), '--output', str(OUT), '--start', '30', '--limit', '270'], cwd=PROJECT, check=True)
print('Saved:', OUT)
